# Assignment 04 — Miền CIFAR-10 · Notebook 01: CNN 2D cài đặt bằng NumPy thuần

**Học phần:** Phát triển các Hệ thống Thông minh — Học viện Công nghệ Bưu chính Viễn thông
**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 – 2027

---

## Mục tiêu của notebook

Notebook này cài đặt mạng tích chập hai chiều **hoàn toàn bằng NumPy**: không dùng PyTorch, không
dùng TensorFlow, không dùng bất kỳ thư viện tự động vi phân nào. Mọi đạo hàm đều được suy ra bằng
tay từ quy tắc chuỗi rồi hiện thực trực tiếp.

So với phiên bản MNIST ở miền `mnist`, điểm mới ở đây là **ảnh có ba kênh màu**. Sự thay đổi này
nghe có vẻ nhỏ nhưng lại chạm vào đúng chỗ dễ sai nhất của cài đặt `im2col`: thứ tự sắp xếp
$(C, K_H, K_W)$ trong ma trận cột phải khớp chính xác với thứ tự khi làm phẳng bộ lọc, nếu không
thì mạng vẫn chạy, vẫn giảm mất mát, nhưng gradient sai và kết quả kém hơn mức đáng lẽ đạt được
mà không có dấu hiệu báo lỗi nào. Vì vậy báo cáo **chạy lại kiểm chứng gradient bằng sai phân hữu
hạn** trên phiên bản ba kênh chứ không tin vào việc phiên bản một kênh đã đạt.

Hai biến thể bắt buộc theo mục 4 của hợp đồng tích hợp:

| | Đệm biên | Khởi tạo | Lịch learning rate |
|---|---|---|---|
| **Baseline** | `padding = 0` | $\mathcal{N}(0, 1) \times 0{,}01$ | cố định |
| **Improved** | `padding = 1` (same) | He Normal $\sigma = \sqrt{2/(C_{in} K^2)}$ | giảm dần theo bậc thang |

**Ba hình bắt buộc sinh ra ở đây:** `fig_cifar10_scratch_curves.png`,
`fig_cifar10_scratch_confusion.png`, `fig_cifar10_scratch_comparison.png`.

## Ngân sách tính toán, công bố ngay từ đầu

Hợp đồng quy định mỗi notebook chạy dưới 15 phút trên CPU. Mạng tích chập viết bằng NumPy thuần
cho ảnh $32 \times 32 \times 3$ tốn khoảng gấp năm lần chi phí của ảnh MNIST $28 \times 28 \times 1$
ở cùng cấu hình. Để nằm trong ngân sách, hai mô hình NumPy được huấn luyện trên **tập con phân
tầng 10 000 ảnh train và 2 500 ảnh validation** thay vì toàn bộ 40 000 / 10 000.

Điều này **không** ảnh hưởng tới tính công bằng của phần đánh giá: cả hai mô hình vẫn được đánh giá
trên **trọn vẹn 10 000 ảnh của tập kiểm thử gốc**. Chênh lệch giữa nhóm NumPy và nhóm framework ở
notebook 02 vì thế là tổng hợp của hai yếu tố — kiến trúc và lượng dữ liệu huấn luyện — và báo cáo
sẽ nói rõ điều đó ở mọi chỗ so sánh, đồng thời ghi vào khóa `numpy_subset` và `notes` của tệp
metrics.

## 1. Nhập thư viện và cấu hình

In [ ]:
import os, json, time, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             classification_report)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/cifar10.npz'
FIG_DIR   = '../reports/figures'
REP_DIR   = '../reports'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

CLASS_VI = ['máy bay', 'ô tô', 'chim', 'mèo', 'hươu',
            'chó', 'ếch', 'ngựa', 'tàu thủy', 'xe tải']

# Ngân sách tính toán cho mô hình NumPy thuần (xem phần mở đầu)
N_SUB_TRAIN = 10000
N_SUB_VAL   = 2500
EPOCHS      = 12
BATCH_SIZE  = 64

print('NumPy', np.__version__)
print(f'Tập con NumPy: train={N_SUB_TRAIN:,}, val={N_SUB_VAL:,}, '
      f'epochs={EPOCHS}, batch={BATCH_SIZE}')

## 2. Nạp dữ liệu, chia tập và chuẩn hóa

Ba bước, theo đúng quy định của hợp đồng tích hợp:

1. `train_test_split(test_size=0.2, stratify=y, random_state=42)` trên 50 000 ảnh huấn luyện để
   tách ra 40 000 train và 10 000 validation. Tập kiểm thử 10 000 ảnh gốc **không bao giờ** tham
   gia vào việc chọn epoch hay chọn siêu tham số.
2. Chuẩn hóa $x/255$ rồi trừ trung bình, chia độ lệch chuẩn **theo từng kênh màu**, với hằng số
   học từ nhánh train.
3. Chuyển bố cục **HWC $\to$ CHW**, tức $(N, 32, 32, 3) \to (N, 3, 32, 32)$. Cài đặt `im2col`
   bên dưới giả định trục kênh nằm ở vị trí thứ hai, giống quy ước của PyTorch.

Sau đó lấy tập con phân tầng cho mô hình NumPy. Phân tầng là bắt buộc: nếu lấy mẫu ngẫu nhiên
đơn thuần trên 10 000 ảnh thì mỗi lớp sẽ dao động quanh 1 000 ảnh với sai số vài chục, đủ để làm
nhiễu phép so sánh `per_class_accuracy` về sau.

In [ ]:
assert os.path.exists(DATA_PATH), f'Không tìm thấy {DATA_PATH}'
_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)
print(f'Dữ liệu gốc: train {x_train_raw.shape}, test {x_test_raw.shape}')

# --- Chia train / validation có phân tầng ---
idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)
x_tr_u8, y_tr_full = x_train_raw[idx_tr], y_train_raw[idx_tr]
x_va_u8, y_va_full = x_train_raw[idx_va], y_train_raw[idx_va]

# --- Hằng số chuẩn hóa theo từng kênh, học từ nhánh train ---
# Tích lũy BẮT BUỘC ở float64: cộng dồn 41 triệu số hạng ở float32 làm trung bình bão hòa
# và cho ra ba giá trị trùng nhau, một hiện vật làm tròn đã được phân tích ở notebook 00.
MEAN_C = np.array([x_tr_u8[:, :, :, c].mean(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
STD_C  = np.array([x_tr_u8[:, :, :, c].std(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
print(f'MEAN theo kênh (R, G, B) = {[round(float(v), 8) for v in MEAN_C]}')
print(f'STD  theo kênh (R, G, B) = {[round(float(v), 8) for v in STD_C]}')
print(f'(học từ {len(idx_tr):,} ảnh nhánh train, tích lũy ở float64)')


def preprocess(x_u8):
    '''uint8 (N,32,32,3) HWC -> float32 (N,3,32,32) CHW đã chuẩn hóa theo từng kênh.'''
    x = x_u8.astype(np.float32) / 255.0
    x = (x - MEAN_C) / STD_C                 # phát sóng trên trục kênh cuối cùng
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))


X_tr_full = preprocess(x_tr_u8)
X_va_full = preprocess(x_va_u8)
X_te      = preprocess(x_test_raw)
y_te      = y_test_raw
print()
print(f'X_tr_full {X_tr_full.shape} | X_va_full {X_va_full.shape} | X_te {X_te.shape}')

# --- Tập con phân tầng cho mô hình NumPy thuần ---
sub_tr, _ = train_test_split(np.arange(len(y_tr_full)), train_size=N_SUB_TRAIN,
                             stratify=y_tr_full, random_state=RANDOM_SEED)
sub_va, _ = train_test_split(np.arange(len(y_va_full)), train_size=N_SUB_VAL,
                             stratify=y_va_full, random_state=RANDOM_SEED)
X_tr, y_tr = X_tr_full[sub_tr], y_tr_full[sub_tr]
X_va, y_va = X_va_full[sub_va], y_va_full[sub_va]
del X_tr_full, X_va_full

print()
print(f'Tập con train NumPy : {X_tr.shape}  phân phối lớp {np.bincount(y_tr, minlength=10).tolist()}')
print(f'Tập con val   NumPy : {X_va.shape}  phân phối lớp {np.bincount(y_va, minlength=10).tolist()}')
print(f'Tập kiểm thử đầy đủ : {X_te.shape} phân phối lớp {np.bincount(y_te, minlength=10).tolist()}')
print()
print(f'Khoảng giá trị sau chuẩn hóa: [{X_tr.min():.4f}, {X_tr.max():.4f}]')
print(f'Trung bình / độ lệch chuẩn tập con train sau chuẩn hóa: '
      f'{X_tr.mean(dtype=np.float64):.6f} / {X_tr.std(dtype=np.float64):.6f}')
for c, nm in enumerate('RGB'):
    print(f'  kênh {nm}: trung bình={X_tr[:, c].mean(dtype=np.float64):+.6f}  '
          f'độ lệch chuẩn={X_tr[:, c].std(dtype=np.float64):.6f}')

**Diễn giải.** Tập con giữ nguyên tính cân bằng: đúng 1 000 ảnh mỗi lớp ở tập con train và 250 ảnh
mỗi lớp ở tập con validation. Sau chuẩn hóa, mỗi kênh có trung bình rất gần 0 và độ lệch chuẩn rất
gần 1 khi tính trên nhánh train đầy đủ; con số của tập con lệch đi chút ít vì nó chỉ là một phần
tư dữ liệu, điều này hoàn toàn bình thường.

Lưu ý rằng trung bình theo từng kênh sau chuẩn hóa không bằng 0 tuyệt đối. Nguyên nhân là hằng số
$\mu_c, \sigma_c$ được học trên 40 000 ảnh nhưng ở đây ta đo trên 10 000 ảnh của tập con. Đây
chính là hành vi đúng: hằng số chuẩn hóa phải cố định theo nhánh train đầy đủ, không được tính
lại theo từng tập con, nếu không thì hai mô hình sẽ dùng hai phép biến đổi khác nhau và không so
sánh được với nhau.

## 3. Cơ sở toán học: phép tích chập hai chiều nhiều kênh

Với ảnh đầu vào $X \in \mathbb{R}^{C_{in} \times H \times W}$ và bộ lọc
$W \in \mathbb{R}^{C_{out} \times C_{in} \times K \times K}$, phép tích chập (chính xác hơn là
tương quan chéo, quy ước của mọi thư viện học sâu) cho đầu ra tại kênh $f$, vị trí $(i, j)$:

$$Z_{f,i,j} = b_f + \sum_{c=0}^{C_{in}-1} \sum_{u=0}^{K-1} \sum_{v=0}^{K-1}
              W_{f,c,u,v} \cdot X_{c,\, i \cdot s + u - p,\; j \cdot s + v - p}$$

trong đó $s$ là bước trượt (stride) và $p$ là độ đệm biên (padding). Kích thước đầu ra:

$$H_{out} = \left\lfloor \frac{H + 2p - K}{s} \right\rfloor + 1$$

Điểm mấu chốt của ảnh màu nằm ở **tổng theo $c$**: mỗi bộ lọc không nhìn một kênh mà nhìn đồng
thời cả ba, rồi cộng ba đáp ứng lại thành một con số duy nhất. Do đó một bộ lọc $3 \times 3$ trên
ảnh RGB có $3 \times 3 \times 3 = 27$ trọng số chứ không phải 9, và nó có khả năng học các đặc
trưng **phụ thuộc màu** — ví dụ biên giữa vùng xanh lam và vùng xám, thứ thường xuất hiện ở đường
chân trời biển.

### Vì sao phải dùng `im2col`

Nếu hiện thực trực tiếp bốn vòng lặp lồng nhau theo công thức trên, Python phải thực hiện
$N \cdot C_{out} \cdot H_{out} \cdot W_{out} \cdot C_{in} \cdot K^2$ phép nhân ở mức trình thông
dịch. Với một lô 64 ảnh CIFAR-10 qua tầng tích chập đầu tiên, con số đó là khoảng 28 triệu phép
nhân — mỗi phép mất hàng trăm nano giây trong Python thuần, tức hàng chục giây cho **một lô**.

Kỹ thuật `im2col` biến bài toán thành một phép nhân ma trận duy nhất. Mọi khối cục bộ
$C_{in} \times K \times K$ được trải thành một cột dài $C_{in} K^2$; ghép tất cả các cột lại được
ma trận $X_{col} \in \mathbb{R}^{C_{in}K^2 \times N H_{out} W_{out}}$. Khi đó:

$$Z_{row} = W_{row} X_{col} + b, \qquad
W_{row} \in \mathbb{R}^{C_{out} \times C_{in}K^2}$$

Toàn bộ tầng tích chập trở thành một lệnh GEMM, được thư viện BLAS bên dưới NumPy thực thi ở tốc
độ mã máy. Cái giá phải trả là bộ nhớ: mỗi điểm ảnh bị nhân bản tối đa $K^2$ lần trong ma trận
cột. Đây là đánh đổi chuẩn mực mà mọi thư viện học sâu đều chấp nhận.

**Thứ tự sắp xếp là chỗ dễ sai nhất.** Hàng thứ $r$ của $X_{col}$ tương ứng với bộ ba
$(c, u, v)$ theo thứ tự "kênh chậm nhất, cột nhanh nhất", tức $r = c K^2 + u K + v$. Khi làm phẳng
bộ lọc bằng `W.reshape(C_out, -1)`, NumPy cũng duyệt theo thứ tự C-order là $(c, u, v)$. Hai thứ
tự này **phải** trùng nhau. Với ảnh một kênh, $c$ luôn bằng 0 nên một cài đặt sai thứ tự vẫn cho
kết quả đúng; với ba kênh thì nó sai ngay. Đó là lý do phần kiểm chứng bên dưới được viết riêng
cho trường hợp nhiều kênh.

In [ ]:
def get_im2col_indices(x_shape, KH, KW, padding=1, stride=1):
    '''Dựng ba mảng chỉ số (k, i, j) mô tả vị trí kênh / hàng / cột của mọi phần tử
    trong ma trận cột. Chỉ phụ thuộc hình dạng nên tính một lần là dùng lại được.'''
    N, C, H, W = x_shape
    out_h = (H + 2 * padding - KH) // stride + 1
    out_w = (W + 2 * padding - KW) // stride + 1

    # offset hàng trong cửa sổ: lặp cho đủ KW cột rồi nhân bản cho C kênh
    i0 = np.tile(np.repeat(np.arange(KH), KW), C)
    # vị trí hàng của từng cửa sổ trên ảnh
    i1 = stride * np.repeat(np.arange(out_h), out_w)
    # offset cột trong cửa sổ
    j0 = np.tile(np.arange(KW), KH * C)
    j1 = stride * np.tile(np.arange(out_w), out_h)

    i = i0.reshape(-1, 1) + i1.reshape(1, -1)      # (C*KH*KW, out_h*out_w)
    j = j0.reshape(-1, 1) + j1.reshape(1, -1)
    # k thay đổi CHẬM NHẤT: hàng r của ma trận cột ứng với (c, u, v), r = c*K*K + u*K + v
    k = np.repeat(np.arange(C), KH * KW).reshape(-1, 1)
    return k.astype(np.intp), i.astype(np.intp), j.astype(np.intp), out_h, out_w


def im2col_indices(x, KH, KW, padding=1, stride=1):
    '''Trải mọi khối cục bộ thành cột. Trả về ma trận (C*KH*KW, N*out_h*out_w).'''
    p = padding
    x_padded = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant') if p > 0 else x
    k, i, j, out_h, out_w = get_im2col_indices(x.shape, KH, KW, padding, stride)
    cols = x_padded[:, k, i, j]                    # (N, C*KH*KW, out_h*out_w)
    C = x.shape[1]
    cols = cols.transpose(1, 2, 0).reshape(C * KH * KW, -1)
    return cols, out_h, out_w


def col2im_indices(cols, x_shape, KH, KW, padding=1, stride=1):
    '''Phép ngược của im2col: CỘNG DỒN gradient về đúng vị trí điểm ảnh gốc.
    Phải cộng dồn chứ không ghi đè, vì một điểm ảnh tham gia nhiều cửa sổ trượt.'''
    N, C, H, W = x_shape
    p = padding
    x_padded = np.zeros((N, C, H + 2 * p, W + 2 * p), dtype=cols.dtype)
    k, i, j, out_h, out_w = get_im2col_indices(x_shape, KH, KW, padding, stride)
    cols_reshaped = cols.reshape(C * KH * KW, -1, N).transpose(2, 0, 1)
    np.add.at(x_padded, (slice(None), k, i, j), cols_reshaped)
    return x_padded if p == 0 else x_padded[:, :, p:-p, p:-p]


# Kiểm tra hình dạng trên ảnh ba kênh
_x = np.arange(1 * 3 * 4 * 4, dtype=np.float64).reshape(1, 3, 4, 4)
_cols, _oh, _ow = im2col_indices(_x, 3, 3, padding=0, stride=1)
print('Đầu vào ba kênh :', _x.shape)
print('Ma trận cột     :', _cols.shape, '  kỳ vọng (C*K*K, N*out_h*out_w) = (27, 4)')
print('out_h, out_w    :', _oh, _ow)
print()
print('Cột đầu tiên, tách lại thành (3, 3, 3) — khối 3x3 góc trên trái của cả ba kênh:')
print(_cols[:, 0].reshape(3, 3, 3).astype(int))
print()
print('Đối chiếu trực tiếp với dữ liệu gốc x[0, :, 0:3, 0:3]:')
print(_x[0, :, 0:3, 0:3].astype(int))
print('Khớp hoàn toàn:', np.array_equal(_cols[:, 0].reshape(3, 3, 3), _x[0, :, 0:3, 0:3]))

**Diễn giải.** Cột đầu tiên của ma trận cột, khi tách lại thành khối $(3, 3, 3)$, trùng khít với
lát cắt `x[0, :, 0:3, 0:3]` của tensor gốc. Điều này chứng minh thứ tự sắp xếp $(c, u, v)$ trong
`im2col` đúng như đã phân tích: 9 phần tử đầu thuộc kênh 0, 9 phần tử tiếp theo thuộc kênh 1, 9
phần tử cuối thuộc kênh 2.

### Đối chiếu với một cài đặt tham chiếu viết bằng vòng lặp tường minh

Phép kiểm tra hình dạng ở trên chưa đủ. Báo cáo viết thêm một cài đặt tham chiếu chậm nhưng
trong suốt — bốn vòng lặp lồng nhau bám sát công thức toán — rồi so sánh kết quả của nó với kết
quả của đường `im2col` + GEMM trên dữ liệu ngẫu nhiên nhiều kênh. Nếu thứ tự kênh bị hoán vị ở
bất kỳ đâu, hai kết quả sẽ lệch nhau ngay.

In [ ]:
def naive_conv2d(x, W, b, padding=0, stride=1):
    '''Cài đặt tham chiếu: bám sát đúng công thức toán, chậm nhưng minh bạch.
    Dùng riêng cho mục đích kiểm chứng, không dùng trong huấn luyện.'''
    N, C, H, Wd = x.shape
    C_out, _, KH, KW = W.shape
    p = padding
    xp = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant') if p else x
    out_h = (H + 2 * p - KH) // stride + 1
    out_w = (Wd + 2 * p - KW) // stride + 1
    out = np.zeros((N, C_out, out_h, out_w), dtype=x.dtype)
    for n in range(N):
        for f in range(C_out):
            for i in range(out_h):
                for j in range(out_w):
                    acc = 0.0
                    for c in range(C):          # TỔNG THEO KÊNH: điểm mấu chốt của ảnh màu
                        for u in range(KH):
                            for v in range(KW):
                                acc += W[f, c, u, v] * xp[n, c, i*stride+u, j*stride+v]
                    out[n, f, i, j] = acc + b[f]
    return out


rng_chk = np.random.default_rng(RANDOM_SEED)
x_chk = rng_chk.standard_normal((2, 3, 6, 6))
W_chk = rng_chk.standard_normal((4, 3, 3, 3))
b_chk = rng_chk.standard_normal(4)

t0 = time.time()
ref = naive_conv2d(x_chk, W_chk, b_chk, padding=1, stride=1)
t_naive = time.time() - t0

t0 = time.time()
cols, oh, ow = im2col_indices(x_chk, 3, 3, padding=1, stride=1)
fast = (W_chk.reshape(4, -1) @ cols + b_chk.reshape(-1, 1))
fast = fast.reshape(4, oh, ow, 2).transpose(3, 0, 1, 2)
t_fast = time.time() - t0

print(f'Cài đặt tham chiếu (vòng lặp) : {ref.shape}, chạy {t_naive*1000:.2f} ms')
print(f'Cài đặt im2col + GEMM         : {fast.shape}, chạy {t_fast*1000:.2f} ms')
print(f'Tăng tốc trên ví dụ tí hon này: {t_naive/max(t_fast,1e-9):.1f} lần')
print()
print(f'Sai lệch tuyệt đối lớn nhất giữa hai cài đặt: {np.abs(ref - fast).max():.3e}')
print('Hai cài đặt đồng nhất (sai số cỡ epsilon máy):', np.allclose(ref, fast, atol=1e-12))
print()

# Một ví dụ số nhỏ tính được bằng tay để minh họa phép cộng theo kênh
X_hand = np.zeros((1, 3, 2, 2))
X_hand[0, 0] = [[1., 2.], [3., 4.]]       # kênh đỏ
X_hand[0, 1] = [[0., 1.], [1., 0.]]       # kênh lục
X_hand[0, 2] = [[2., 0.], [0., 2.]]       # kênh lam
W_hand = np.ones((1, 3, 2, 2))            # bộ lọc toàn số 1: cộng mọi phần tử của cả ba kênh
b_hand = np.zeros(1)
cols_h, _, _ = im2col_indices(X_hand, 2, 2, padding=0, stride=1)
z_hand = float((W_hand.reshape(1, -1) @ cols_h + b_hand)[0, 0])
tong_tay = (1+2+3+4) + (0+1+1+0) + (2+0+0+2)
print('Ví dụ tính tay — bộ lọc toàn số 1 trên ảnh 2x2 ba kênh:')
print(f'  kênh đỏ  cộng lại = {1+2+3+4}')
print(f'  kênh lục cộng lại = {0+1+1+0}')
print(f'  kênh lam cộng lại = {2+0+0+2}')
print(f'  tổng ba kênh tính tay = {tong_tay}')
print(f'  kết quả của cài đặt    = {z_hand}')
print('  khớp:', z_hand == tong_tay)

**Diễn giải.** Hai phép kiểm chứng độc lập cùng cho kết quả khớp. Phép thứ nhất so cài đặt nhanh
với cài đặt tham chiếu viết theo đúng công thức toán trên dữ liệu ngẫu nhiên ba kênh: sai lệch ở
mức epsilon của số thực chấm động, tức hai cài đặt tính cùng một hàm. Phép thứ hai dùng một ví dụ
đủ nhỏ để kiểm tra bằng tay, cho thấy tầng tích chập thực sự **cộng đáp ứng của cả ba kênh** chứ
không xử lý từng kênh riêng rẽ.

Cần nói thêm về con số tăng tốc: trên ví dụ tí hon $2 \times 3 \times 6 \times 6$ này, ưu thế của
`im2col` đã thấy rõ, và khoảng cách giãn ra rất nhanh khi kích thước tăng. Ở quy mô thật của
notebook — lô 64 ảnh $3 \times 32 \times 32$ — cài đặt vòng lặp sẽ mất hàng chục giây cho một lô,
tức hoàn toàn không dùng được.

## 4. Lan truyền ngược qua tầng tích chập

Đặt $Z_{row} = W_{row} X_{col} + b$. Với $G = \partial L / \partial Z_{row}$ nhận được từ tầng phía
sau, ba đạo hàm cần tính là:

$$\frac{\partial L}{\partial W_{row}} = G X_{col}^{\top}, \qquad
  \frac{\partial L}{\partial b_f} = \sum_{n,i,j} G_{f,(n,i,j)}, \qquad
  \frac{\partial L}{\partial X_{col}} = W_{row}^{\top} G$$

Hai công thức đầu là đạo hàm chuẩn của phép nhân ma trận. Công thức thứ ba cho gradient theo ma
trận cột, nhưng cái ta cần là gradient theo **ảnh gốc**. Vì mỗi điểm ảnh xuất hiện trong nhiều cột
khác nhau (nó thuộc nhiều cửa sổ trượt), quy tắc chuỗi bắt buộc phải **cộng dồn** mọi đóng góp:

$$\frac{\partial L}{\partial X_{c,y,x}} = \sum_{\text{mọi cửa sổ chứa } (c,y,x)}
   \frac{\partial L}{\partial X_{col}}\bigg|_{\text{vị trí tương ứng}}$$

Đó chính là việc mà `col2im_indices` làm, và là lý do nó dùng `np.add.at` thay cho phép gán thông
thường. Gán thông thường sẽ chỉ giữ lại đóng góp cuối cùng và làm gradient sai một cách âm thầm —
mạng vẫn học được đôi chút nên lỗi rất khó phát hiện nếu không có kiểm chứng gradient.

In [ ]:
class Conv2D:
    '''Tầng tích chập hai chiều nhiều kênh, vector hóa hoàn toàn bằng im2col / col2im.'''

    def __init__(self, C_in, C_out, K=3, padding=1, stride=1, init='scaled', seed=0,
                 dtype=np.float32):
        rng = np.random.default_rng(seed)
        if init == 'he':
            # He Normal: sigma = sqrt(2 / fan_in), fan_in = C_in * K * K
            sigma = np.sqrt(2.0 / (C_in * K * K))
        else:
            sigma = 0.01                      # khởi tạo randn * 0.01 của Baseline
        self.W = (rng.standard_normal((C_out, C_in, K, K)) * sigma).astype(dtype)
        self.b = np.zeros(C_out, dtype=dtype)
        self.K, self.padding, self.stride = K, padding, stride
        self.C_in, self.C_out = C_in, C_out
        self.init_sigma = float(sigma)

    def params(self):
        return [('W', self.W), ('b', self.b)]

    def grads(self):
        return [('W', self.dW), ('b', self.db)]

    def forward(self, x):
        self.x_shape = x.shape
        cols, oh, ow = im2col_indices(x, self.K, self.K, self.padding, self.stride)
        self.cols = cols
        W_row = self.W.reshape(self.C_out, -1)             # (C_out, C_in*K*K)
        out = W_row @ cols + self.b.reshape(-1, 1)         # một lệnh GEMM duy nhất
        out = out.reshape(self.C_out, oh, ow, x.shape[0]).transpose(3, 0, 1, 2)
        return out

    def backward(self, dout):
        # dout: (N, C_out, oh, ow)
        self.db = dout.sum(axis=(0, 2, 3))
        G = dout.transpose(1, 2, 3, 0).reshape(self.C_out, -1)   # (C_out, N*oh*ow)
        self.dW = (G @ self.cols.T).reshape(self.W.shape)        # dL/dW_row = G X_col^T
        W_row = self.W.reshape(self.C_out, -1)
        dcols = W_row.T @ G                                      # dL/dX_col = W_row^T G
        return col2im_indices(dcols, self.x_shape, self.K, self.K,
                              self.padding, self.stride)

    def out_size(self, H):
        return (H + 2 * self.padding - self.K) // self.stride + 1


print('Lớp Conv2D đã được định nghĩa.')
_c = Conv2D(3, 16, 3, padding=1, init='he', seed=1, dtype=np.float64)
print(f'Kích thước W : {_c.W.shape}  ({_c.W.size} trọng số = 16 bộ lọc x 3 kênh x 3 x 3)')
print(f'sigma khởi tạo He = {_c.init_sigma:.8f}  |  '
      f'sqrt(2/27) = {np.sqrt(2/27):.8f}')
print(f'Kích thước ảnh: 32 -> {_c.out_size(32)} (đệm biên same giữ nguyên kích thước)')
_c0 = Conv2D(3, 16, 3, padding=0, init='scaled', seed=1, dtype=np.float64)
print(f'Không đệm biên: 32 -> {_c0.out_size(32)} (mất 2 điểm ảnh mỗi chiều)')

**Diễn giải.** Bộ lọc đầu tiên của phiên bản ba kênh có $16 \times 3 \times 3 \times 3 = 432$
trọng số, gấp ba lần con số 144 của phiên bản MNIST một kênh cùng cấu hình. Đây là chi phí trực
tiếp của việc xử lý ảnh màu: mỗi bộ lọc phải học một khuôn mẫu riêng cho từng kênh rồi cộng lại.

Giá trị $\sigma$ của He Normal cũng thay đổi theo: $\sqrt{2/27} \approx 0{,}2722$ cho ảnh ba kênh,
so với $\sqrt{2/9} \approx 0{,}4714$ cho ảnh một kênh. Công thức tự động thu nhỏ biên độ khởi tạo
khi số đầu vào tăng, đúng với mục đích của nó là giữ phương sai tín hiệu ổn định khi truyền qua
các tầng.

## 5. Các tầng còn lại

**ReLU.** $a = \max(0, z)$, đạo hàm $\partial a / \partial z = \mathbb{1}[z > 0]$. Cài đặt chỉ cần
nhớ mặt nạ dấu ở lượt thuận.

**Max pooling $2 \times 2$ bước 2.** Lấy giá trị lớn nhất trong mỗi ô $2 \times 2$ không chồng
lấn. Gradient được **định tuyến** nguyên vẹn về đúng ô đã thắng, các ô còn lại nhận 0:

$$\frac{\partial L}{\partial x_{u,v}} =
  \begin{cases} \dfrac{\partial L}{\partial y} & \text{nếu } (u,v) = \arg\max \\[4pt]
  0 & \text{ngược lại}\end{cases}$$

Ở đây ta gộp trục $N$ và trục $C$ lại thành một để pooling độc lập trên từng kênh, rồi dùng lại
đúng cơ chế `im2col` với `padding = 0, stride = 2`.

**Flatten.** Chỉ đổi hình dạng, lượt ngược đổi ngược lại.

**Dense.** $z = xW + b$, với $\partial L/\partial W = x^{\top} \delta$,
$\partial L/\partial b = \sum_n \delta_n$, $\partial L/\partial x = \delta W^{\top}$.

In [ ]:
class ReLU:
    def params(self): return []
    def grads(self):  return []
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, d):
        return d * self.mask


class MaxPool2D:
    '''Max pooling 2x2 bước 2, gradient định tuyến theo argmax, vector hóa qua im2col.'''
    def __init__(self, size=2, stride=2):
        self.size, self.stride = size, stride
    def params(self): return []
    def grads(self):  return []

    def forward(self, x):
        N, C, H, W = x.shape
        k, s = self.size, self.stride
        oh, ow = (H - k) // s + 1, (W - k) // s + 1
        self.x_shape = x.shape
        x_r = x.reshape(N * C, 1, H, W)               # gộp N và C để pool độc lập từng kênh
        cols, _, _ = im2col_indices(x_r, k, k, padding=0, stride=s)   # (k*k, N*C*oh*ow)
        self.cols_shape = cols.shape
        self.arg = np.argmax(cols, axis=0)            # ghi nhớ vị trí thắng cuộc
        out = cols[self.arg, np.arange(cols.shape[1])]
        return out.reshape(oh, ow, N * C).transpose(2, 0, 1).reshape(N, C, oh, ow)

    def backward(self, dout):
        N, C, H, W = self.x_shape
        k, s = self.size, self.stride
        oh, ow = dout.shape[2], dout.shape[3]
        dcols = np.zeros(self.cols_shape, dtype=dout.dtype)
        d_flat = dout.reshape(N * C, oh, ow).transpose(1, 2, 0).ravel()
        dcols[self.arg, np.arange(dcols.shape[1])] = d_flat   # chỉ ô argmax nhận gradient
        dx = col2im_indices(dcols, (N * C, 1, H, W), k, k, padding=0, stride=s)
        return dx.reshape(N, C, H, W)


class Flatten:
    def params(self): return []
    def grads(self):  return []
    def forward(self, x):
        self.shape = x.shape
        return x.reshape(x.shape[0], -1)
    def backward(self, d):
        return d.reshape(self.shape)


class Dense:
    def __init__(self, n_in, n_out, init='scaled', seed=0, dtype=np.float32):
        rng = np.random.default_rng(seed)
        sigma = np.sqrt(2.0 / n_in) if init == 'he' else 0.01
        self.W = (rng.standard_normal((n_in, n_out)) * sigma).astype(dtype)
        self.b = np.zeros(n_out, dtype=dtype)
        self.init_sigma = float(sigma)
    def params(self): return [('W', self.W), ('b', self.b)]
    def grads(self):  return [('W', self.dW), ('b', self.db)]
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    def backward(self, d):
        self.dW = self.x.T @ d
        self.db = d.sum(axis=0)
        return d @ self.W.T


# Kiểm chứng định tuyến gradient của MaxPool trên ảnh HAI kênh, tính tay được
xp = np.zeros((1, 2, 4, 4))
xp[0, 0] = [[1., 5., 2., 0.], [3., 2., 1., 4.], [0., 1., 9., 2.], [7., 3., 2., 2.]]
xp[0, 1] = [[8., 0., 1., 1.], [0., 2., 3., 0.], [4., 4., 0., 6.], [1., 5., 2., 1.]]
mp = MaxPool2D(2, 2)
yp = mp.forward(xp)
print('Kênh 0 đầu vào:\n', xp[0, 0])
print('Kênh 0 sau pool (kỳ vọng [[5,4],[7,9]]):\n', yp[0, 0])
print()
print('Kênh 1 đầu vào:\n', xp[0, 1])
print('Kênh 1 sau pool (kỳ vọng [[8,3],[5,6]]):\n', yp[0, 1])
print()
gp = mp.backward(np.ones_like(yp))
print('Tổng gradient truyền ngược =', gp.sum(),
      '(phải bằng số ô đầu ra = 2 kênh x 4 ô = 8)')
print('Gradient kênh 0 (số 1 tại đúng bốn ô thắng cuộc):\n', gp[0, 0])
print('Hai kênh được pool độc lập:',
      not np.array_equal(yp[0, 0], yp[0, 1]))

**Diễn giải.** Kết quả pooling của hai kênh khác nhau và mỗi kênh đều khớp với giá trị tính tay,
chứng tỏ thao tác gộp trục $N$ và $C$ không làm lẫn dữ liệu giữa các kênh. Tổng gradient truyền
ngược bằng đúng 8, tức mỗi ô đầu ra gửi trọn vẹn một đơn vị gradient về đúng một ô đầu vào — đúng
như tính chất định tuyến của max pooling.

## 6. Hàm mất mát và thuật toán tối ưu

**Softmax.** $\hat{y}_k = e^{z_k} / \sum_j e^{z_j}$. Cài đặt trừ đi giá trị lớn nhất của mỗi hàng
trước khi lấy lũy thừa; phép trừ này không đổi kết quả về mặt toán học nhưng tránh tràn số khi
logit lớn.

**Entropy chéo.** $L = -\frac{1}{N}\sum_{n} \log \hat{y}_{n, y_n}$.

**Gradient hợp nhất.** Đạo hàm của entropy chéo theo logit, sau khi rút gọn qua softmax, có dạng
đặc biệt gọn:

$$\frac{\partial L}{\partial z_{n,k}} = \frac{1}{N}\left(\hat{y}_{n,k} - \mathbb{1}[k = y_n]\right)$$

Việc gộp hai tầng thành một biểu thức vừa nhanh hơn vừa ổn định hơn nhiều so với tính riêng đạo
hàm của softmax rồi nhân với đạo hàm của log.

**Adam.** Duy trì trung bình trượt bậc một và bậc hai của gradient, có hiệu chỉnh độ chệch:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}, \quad
  \theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
def softmax(z):
    '''Softmax ổn định số: trừ cực đại theo từng hàng trước khi lấy lũy thừa.'''
    z_shift = z - z.max(axis=1, keepdims=True)
    e = np.exp(z_shift)
    return e / e.sum(axis=1, keepdims=True)


def softmax_cross_entropy(z, y):
    '''Trả về (mất mát trung bình, dL/dz, xác suất dự đoán).
    Gradient hợp nhất: dL/dz = (y_hat - y_onehot) / N.'''
    p = softmax(z)
    N = z.shape[0]
    loss = float(-np.log(np.clip(p[np.arange(N), y], 1e-12, None)).mean())
    dz = p.copy()
    dz[np.arange(N), y] -= 1.0
    dz /= N
    return loss, dz, p


class Adam:
    def __init__(self, layers, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.layers = layers
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [{n: np.zeros_like(p) for n, p in l.params()} for l in layers]
        self.v = [{n: np.zeros_like(p) for n, p in l.params()} for l in layers]
        self.t = 0

    def step(self, lr=None):
        lr = self.lr if lr is None else lr
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t
        bc2 = 1.0 - self.b2 ** self.t
        for li, layer in enumerate(self.layers):
            gd = dict(layer.grads())
            for name, p in layer.params():
                g = gd[name]
                self.m[li][name] = self.b1 * self.m[li][name] + (1 - self.b1) * g
                self.v[li][name] = self.b2 * self.v[li][name] + (1 - self.b2) * (g * g)
                m_hat = self.m[li][name] / bc1
                v_hat = self.v[li][name] / bc2
                p -= lr * m_hat / (np.sqrt(v_hat) + self.eps)   # cập nhật tại chỗ


z_big = np.array([[1000.0, 999.0, 998.0]])
with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(z_big) / np.exp(z_big).sum(axis=1, keepdims=True)
print('Softmax cài ngây thơ trên logit lớn :', naive)
print('Softmax có trừ cực đại              :', softmax(z_big))
print('Tổng xác suất                       :', softmax(z_big).sum())
print()
z0 = np.zeros((1, 10))
l0, _, _ = softmax_cross_entropy(z0, np.array([3]))
print(f'Mất mát khi mọi logit bằng nhau: {l0:.10f}  |  log(10) = {math.log(10):.10f}')
print('Đây là mức mất mát kỳ vọng của một mạng chưa học gì trên bài toán 10 lớp cân bằng.')

## 7. Kiến trúc hai biến thể

Theo mục 4 hợp đồng tích hợp, biến thể CIFAR-10 dùng kênh $16 \to 32$ và tầng ẩn 128 nơ-ron:

```
Conv2D(16, K=3) -> ReLU -> MaxPool(2)
Conv2D(32, K=3) -> ReLU -> MaxPool(2)
Flatten -> Dense(128) -> ReLU -> Dense(10) -> Softmax
```

Khác biệt duy nhất giữa hai biến thể nằm ở ba yếu tố đã nêu ở đầu notebook. Hệ quả về kích thước:

- **Baseline** (`padding = 0`): $32 \to 30 \to 15 \to 13 \to 6$, véc-tơ phẳng $32 \times 6 \times 6 = 1152$.
- **Improved** (`padding = 1`): $32 \to 32 \to 16 \to 16 \to 8$, véc-tơ phẳng $32 \times 8 \times 8 = 2048$.

Đệm biên giữ nguyên kích thước không gian sau mỗi tầng tích chập nên biến thể Improved có véc-tơ
phẳng lớn hơn, kéo theo nhiều tham số hơn ở tầng Dense đầu tiên. Đây là một khác biệt cần thừa
nhận thẳng thắn: Improved thắng Baseline không chỉ nhờ khởi tạo và lịch learning rate mà còn nhờ
nhiều tham số hơn. Báo cáo sẽ ghi rõ cả hai con số tham số khi so sánh, thay vì quy toàn bộ mức
cải thiện cho các thủ thuật tối ưu.

In [ ]:
class CNN:
    '''Mạng tích chập hai chiều cho ảnh ba kênh:
    Conv-ReLU-Pool x2 -> Flatten -> Dense-ReLU -> Dense.'''

    def __init__(self, padding=0, init='scaled', seed=RANDOM_SEED,
                 c_in=3, c1=16, c2=32, n_hidden=128, dtype=np.float32, img=32):
        self.conv1 = Conv2D(c_in, c1, 3, padding, init=init, seed=seed,     dtype=dtype)
        self.conv2 = Conv2D(c1,   c2, 3, padding, init=init, seed=seed + 1, dtype=dtype)
        s1 = self.conv1.out_size(img);  s2 = s1 // 2
        s3 = self.conv2.out_size(s2);   s4 = s3 // 2
        self.flat_dim = c2 * s4 * s4
        self.shape_trace = [img, s1, s2, s3, s4, self.flat_dim]
        self.fc1 = Dense(self.flat_dim, n_hidden, init=init, seed=seed + 2, dtype=dtype)
        self.fc2 = Dense(n_hidden, 10,           init=init, seed=seed + 3, dtype=dtype)
        self.relu1, self.relu2, self.relu3 = ReLU(), ReLU(), ReLU()
        self.pool1, self.pool2 = MaxPool2D(2, 2), MaxPool2D(2, 2)
        self.flatten = Flatten()
        self.seq = [self.conv1, self.relu1, self.pool1,
                    self.conv2, self.relu2, self.pool2,
                    self.flatten, self.fc1, self.relu3, self.fc2]
        self.learnables = [('conv1', self.conv1), ('conv2', self.conv2),
                           ('fc1', self.fc1),     ('fc2', self.fc2)]

    def forward(self, x):
        for layer in self.seq:
            x = layer.forward(x)
        return x

    def backward(self, dz):
        for layer in reversed(self.seq):
            dz = layer.backward(dz)
        return dz

    def n_params(self):
        return int(sum(p.size for _, l in self.learnables for _, p in l.params()))


_nb = CNN(padding=0, init='scaled')
_ni = CNN(padding=1, init='he')
for tag, net in [('Baseline', _nb), ('Improved', _ni)]:
    t = net.shape_trace
    print(f'{tag:9s}: 32 -> conv1 {t[1]} -> pool {t[2]} -> conv2 {t[3]} -> pool {t[4]} '
          f'-> phẳng {t[5]:,}  |  tham số = {net.n_params():,}')
print()
print('Phân rã tham số của biến thể Improved:')
for name, l in _ni.learnables:
    tot = sum(p.size for _, p in l.params())
    print(f'  {name:<6s}: {tot:>9,} tham số')
print(f'  {"tổng":<6s}: {_ni.n_params():>9,}')
print()
print(f'Tỉ lệ tham số nằm ở tầng Dense đầu tiên (Improved): '
      f'{100*sum(p.size for _, p in _ni.fc1.params())/_ni.n_params():.1f}%')
print('Nhận xét: phần lớn tham số tập trung ở tầng kết nối đầy đủ chứ không phải ở tầng tích')
print('chập. Đây là đặc điểm chung của các CNN nông, và cũng là động cơ để các kiến trúc hiện')
print('đại thay Flatten + Dense lớn bằng Global Average Pooling.')
del _nb, _ni

## 8. Kiểm chứng gradient bằng sai phân hữu hạn trên phiên bản ba kênh

Đây là phần quan trọng nhất của notebook về mặt phương pháp. Mọi đạo hàm ở trên đều được suy ra
bằng tay; nếu một dấu hoặc một phép hoán vị trục sai, mạng vẫn chạy và mất mát vẫn giảm, nhưng
giảm chậm hơn đáng lẽ, và không có thông báo lỗi nào xuất hiện.

Phép kiểm chứng dùng sai phân hữu hạn **trung tâm**:

$$\frac{\partial L}{\partial \theta_i} \approx
  \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2\varepsilon}$$

Công thức trung tâm có sai số bậc $O(\varepsilon^2)$, tốt hơn hẳn công thức tiến bậc $O(\varepsilon)$.
Đại lượng so sánh là sai số tương đối:

$$\text{rel} = \frac{|g_{\text{giải tích}} - g_{\text{số}}|}
                    {|g_{\text{giải tích}}| + |g_{\text{số}}|}$$

Ngưỡng đánh giá theo thông lệ: dưới $10^{-7}$ là đạt hoàn toàn, dưới $10^{-5}$ là chấp nhận được,
lớn hơn $10^{-3}$ gần như chắc chắn có lỗi cài đặt.

Ba lựa chọn kỹ thuật cần giải thích:

1. **Dùng `float64`.** Ở `float32`, sai số làm tròn của chính phép tính mất mát đã cùng bậc với
   $\varepsilon$, khiến phép kiểm chứng vô nghĩa.
2. **Chọn tọa độ có $|gradient|$ lớn.** Ở những tọa độ gradient gần 0, sai số tương đối rất nhiễu
   mà không nói lên điều gì về tính đúng đắn.
3. **Kiểm trên đủ bốn tensor học được** — `conv1`, `conv2`, `fc1`, `fc2` — chứ không chỉ tầng
   cuối. Lỗi xử lý kênh nếu có sẽ nằm ở `conv1`, tầng duy nhất nhận đầu vào ba kênh.

### Điểm gãy: vì sao một phép kiểm ngây thơ báo sai số lớn giả tạo

Hàm mất mát của mạng này **không khả vi ở mọi nơi**. ReLU có một điểm gãy tại 0, và max pooling
có điểm gãy tại mọi cấu hình mà hai ô trong cùng cửa sổ bằng nhau. Ở lân cận những điểm đó, hàm
là hai đoạn tuyến tính khác độ dốc ghép lại.

Khi tọa độ tham số bị nhiễu, nếu $\theta - \varepsilon$ và $\theta + \varepsilon$ nằm ở **hai
phía khác nhau** của một điểm gãy, thì sai phân hữu hạn đo độ dốc trung bình của hai nhánh, trong
khi lan truyền ngược trả về độ dốc của đúng một nhánh. Hai con số khác nhau, và chúng **phải**
khác nhau — đây không phải lỗi cài đặt mà là hệ quả của việc hàm không khả vi tại đó.

Với ảnh ba kênh $32 \times 32$, mạng có hàng chục nghìn đơn vị ReLU và hàng nghìn cửa sổ pooling,
nên xác suất một phép nhiễu chạm phải ít nhất một điểm gãy cao hơn nhiều so với mạng MNIST nhỏ
hơn. Một phép kiểm không xét điều này sẽ báo sai số tương đối cỡ $10^{-4}$ và khiến người đọc kết
luận nhầm là cài đặt có lỗi.

Vì vậy phép kiểm dưới đây **nhận biết điểm gãy**: ở mỗi tọa độ, nó chụp lại mặt nạ dấu của cả ba
tầng ReLU và vị trí argmax của cả hai tầng max pooling trong lượt thuận $+\varepsilon$ và lượt
thuận $-\varepsilon$. Nếu hai trạng thái khác nhau, tọa độ đó bị **loại bỏ** và được đếm riêng;
nếu giống nhau, hàm trơn trên cả đoạn và phép so sánh là hợp lệ. Số tọa độ bị loại được báo cáo
tường minh chứ không giấu đi.

In [ ]:
def kink_state(net):
    '''Ảnh chụp trạng thái của mọi tầng KHÔNG khả vi trong mạng:
    mặt nạ dấu của ba tầng ReLU và vị trí argmax của hai tầng max pool.'''
    return (net.relu1.mask.copy(), net.relu2.mask.copy(), net.relu3.mask.copy(),
            net.pool1.arg.copy(),  net.pool2.arg.copy())


def same_kink_state(a, b):
    '''True nếu hai lượt thuận đi qua cùng một nhánh của mọi tầng không khả vi.'''
    return all(np.array_equal(u, v) for u, v in zip(a, b))


def gradient_check(net, X, y, eps=1e-5, n_per_tensor=2, max_tries=60, seed=RANDOM_SEED):
    '''So sánh gradient giải tích với gradient sai phân hữu hạn trung tâm trên MỌI tensor
    tham số học được, CÓ LOẠI TRỪ các tọa độ mà phép nhiễu vượt qua một điểm gãy.

    Trả về (loss, rows, n_kink) với n_kink là số tọa độ bị loại vì gặp điểm gãy.'''
    rng = np.random.default_rng(seed)

    # 1. Một lượt thuận + ngược để lấy gradient giải tích
    z = net.forward(X)
    loss, dz, _ = softmax_cross_entropy(z, y)
    net.backward(dz)
    # Sao chép lại ngay: các lượt thuận bổ sung phía dưới sẽ ghi đè bộ đệm nội bộ của tầng
    analytic = {(ln, pn): g.copy() for ln, l in net.learnables for pn, g in l.grads()}

    rows, n_kink = [], 0
    for lname, layer in net.learnables:
        for pname, P in layer.params():
            G = analytic[(lname, pname)]
            flat_g = G.ravel()
            order = np.argsort(-np.abs(flat_g))       # tọa độ có |gradient| lớn nhất trước
            top = order[:max(2, len(order) // 5)]
            # Thứ tự thử: tọa độ dốc nhất trước, sau đó xáo trộn trong nhóm 20% dốc nhất
            trial = [int(order[0])] + [int(v) for v in rng.permutation(top)]
            seen, picked, tried = set(), 0, 0
            for flat_idx in trial:
                if picked >= n_per_tensor or tried >= max_tries:
                    break
                if flat_idx in seen:
                    continue
                seen.add(flat_idx); tried += 1
                idx = np.unravel_index(flat_idx, P.shape)
                old = P[idx]

                P[idx] = old + eps
                lp, _, _ = softmax_cross_entropy(net.forward(X), y)
                st_p = kink_state(net)

                P[idx] = old - eps
                lm, _, _ = softmax_cross_entropy(net.forward(X), y)
                st_m = kink_state(net)

                P[idx] = old                          # phục hồi nguyên trạng

                # Nếu hai lượt thuận đi qua hai nhánh khác nhau của ReLU hoặc MaxPool thì
                # hàm mất mát KHÔNG khả vi trên đoạn [theta-eps, theta+eps]; sai phân hữu hạn
                # ở đó đo một đại lượng khác với đạo hàm, nên tọa độ này phải bị loại.
                if not same_kink_state(st_p, st_m):
                    n_kink += 1
                    continue

                g_num = (lp - lm) / (2 * eps)
                g_ana = float(G[idx])
                rel = abs(g_ana - g_num) / max(1e-12, abs(g_ana) + abs(g_num))
                rows.append((lname, pname, tuple(int(v) for v in idx),
                             g_ana, float(g_num), rel))
                picked += 1
    return loss, rows, n_kink


# Kiểm trên mạng Improved ba kênh, độ chính xác float64, với 8 ảnh CIFAR-10 thật
rng_gc = np.random.default_rng(RANDOM_SEED)
gc_idx = rng_gc.choice(len(X_tr), size=8, replace=False)
X_gc = X_tr[gc_idx].astype(np.float64)
y_gc = y_tr[gc_idx]
print(f'Dữ liệu kiểm chứng: {X_gc.shape} (8 ảnh ba kênh), nhãn {y_gc.tolist()}')
print()

net_gc = CNN(padding=1, init='he', dtype=np.float64, seed=RANDOM_SEED)
t0 = time.time()
loss_gc, rows, n_kink = gradient_check(net_gc, X_gc, y_gc, eps=1e-5, n_per_tensor=2)
t_gc = time.time() - t0

print(f'Mất mát tại điểm kiểm tra: {loss_gc:.10f}   '
      f'(log(10) = {math.log(10):.6f} là mức kỳ vọng khi mạng chưa học)')
print(f'Thời gian kiểm chứng: {t_gc:.1f}s | {len(rows)} tọa độ hợp lệ, '
      f'{n_kink} tọa độ bị loại vì gặp điểm gãy')
print()
hdr = (f"{'Tầng':<7} {'Tham số':<8} {'Chỉ số':<18} {'Gradient giải tích':>22} "
       f"{'Gradient số':>22} {'Sai số tương đối':>18}")
print(hdr); print('-' * len(hdr))
worst = 0.0
for lname, pname, idx, ga, gn, rel in rows:
    worst = max(worst, rel)
    print(f'{lname:<7} {pname:<8} {str(idx):<18} {ga:>22.15f} {gn:>22.15f} {rel:>18.3e}')
print('-' * len(hdr))
print(f'Sai số tương đối lớn nhất trên toàn bộ {len(rows)} phép kiểm tra hợp lệ: {worst:.3e}')
print('Kết luận:', 'ĐẠT (mọi sai số < 1e-7)' if worst < 1e-7 else
      ('CHẤP NHẬN ĐƯỢC (< 1e-5)' if worst < 1e-5 else 'KHÔNG ĐẠT, cài đặt có lỗi'))
print()
conv1_rows = [r for r in rows if r[0] == 'conv1']
print(f'Riêng tầng conv1 — tầng duy nhất nhận đầu vào ba kênh — có {len(conv1_rows)} phép kiểm '
      f'hợp lệ, sai số tương đối lớn nhất {max(r[5] for r in conv1_rows):.3e}.')
print(f'Tỉ lệ tọa độ bị loại vì điểm gãy: '
      f'{100*n_kink/max(1, n_kink+len(rows)):.1f}% số tọa độ đã thử.')

**Diễn giải bảng kiểm chứng gradient.** Cột "gradient giải tích" là kết quả của phép lan truyền
ngược viết tay, cột "gradient số" hoàn toàn độc lập với nó vì chỉ gọi hàm mất mát ở hai điểm lân
cận. Hai cột trùng nhau tới nhiều chữ số có nghĩa trên cả bốn tensor tham số.

Số tọa độ bị loại vì điểm gãy được in ngay dưới bảng. Con số đó khác 0 là điều **phải xảy ra** với
một mạng dùng ReLU và max pooling ở quy mô này, và việc loại chúng ra không phải là cách né tránh:
những tọa độ đó nằm ở chỗ hàm mất mát không có đạo hàm, nên không tồn tại một giá trị đúng nào để
so sánh. Phép kiểm chỉ khẳng định điều nó thực sự kiểm được — rằng trên **mọi tọa độ mà hàm trơn**,
gradient giải tích khớp với gradient số tới mức nhiễu của số học dấu phẩy động.

Dòng dành riêng cho `conv1` là điểm cần chú ý nhất: đây là tầng duy nhất tiếp xúc trực tiếp với ba
kênh màu, nên nếu thứ tự $(c, u, v)$ trong `im2col` lệch khỏi thứ tự làm phẳng bộ lọc, sai số ở
tầng này sẽ bật lên mức $10^{-1}$ thay vì nằm ở mức nhiễu số học. Sai số quan sát được cho thấy
việc mở rộng từ một kênh sang ba kênh là đúng.

Phép kiểm chứng này **không** chứng minh mô hình sẽ đạt độ chính xác cao. Nó chỉ chứng minh một
điều hẹp nhưng thiết yếu: thuật toán tối ưu đang đi xuống theo đúng hướng dốc nhất của hàm mất
mát đã định nghĩa. Mọi kết quả kém sau đó, nếu có, thuộc về kiến trúc hoặc dữ liệu chứ không thuộc
về phép tính đạo hàm.

## 9. Vòng lặp huấn luyện

Hàm huấn luyện dưới đây chọn epoch tốt nhất theo **độ chính xác trên tập kiểm định**, không bao
giờ theo tập kiểm thử. Trọng số của epoch tốt nhất được lưu lại và khôi phục khi kết thúc, tương
đương cơ chế early stopping nhưng vẫn chạy hết số epoch đã định để có đường cong đầy đủ cho hình vẽ.

In [ ]:
def evaluate(net, X, y, batch=256, return_prob=False):
    '''Đánh giá theo lô để không làm tràn bộ nhớ khi chạy trên 10 000 ảnh.'''
    losses, probs = [], []
    for s in range(0, len(X), batch):
        xb, yb = X[s:s + batch], y[s:s + batch]
        z = net.forward(xb)
        l, _, p = softmax_cross_entropy(z, yb)
        losses.append(l * len(xb))
        probs.append(p)
    P = np.concatenate(probs, axis=0)
    loss = float(np.sum(losses) / len(X))
    pred = P.argmax(axis=1)
    acc = float((pred == y).mean())
    return (loss, acc, pred, P) if return_prob else (loss, acc)


def snapshot(net):
    return {name: {pn: p.copy() for pn, p in l.params()} for name, l in net.learnables}


def restore(net, snap):
    for name, l in net.learnables:
        for pn, p in l.params():
            p[...] = snap[name][pn]


def train_model(net, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
                lr0=1e-3, lr_decay=None, decay_every=4, tag='model', seed=RANDOM_SEED):
    '''Huấn luyện một mạng CNN. lr_decay=None nghĩa là learning rate cố định.'''
    rng = np.random.default_rng(seed)
    opt = Adam([l for _, l in net.learnables], lr=lr0)
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_acc, best_epoch, best_snap = -1.0, 0, None
    t_start = time.time()
    n = len(X_tr)

    for ep in range(1, epochs + 1):
        lr_ep = lr0 if lr_decay is None else lr0 * (lr_decay ** ((ep - 1) // decay_every))
        perm = rng.permutation(n)
        run_loss, run_correct, seen = 0.0, 0, 0
        for s in range(0, n, batch):
            bidx = perm[s:s + batch]
            xb, yb = X_tr[bidx], y_tr[bidx]
            z = net.forward(xb)
            loss, dz, p = softmax_cross_entropy(z, yb)
            net.backward(dz)
            opt.step(lr_ep)
            run_loss += loss * len(bidx)
            run_correct += int((p.argmax(axis=1) == yb).sum())
            seen += len(bidx)
        tr_loss, tr_acc = run_loss / seen, run_correct / seen
        va_loss, va_acc = evaluate(net, X_va, y_va)

        hist['train_loss'].append(float(tr_loss)); hist['val_loss'].append(float(va_loss))
        hist['train_acc'].append(float(tr_acc));   hist['val_acc'].append(float(va_acc))
        hist['lr'].append(float(lr_ep))
        star = ''
        if va_acc > best_acc:
            best_acc, best_epoch, best_snap = va_acc, ep, snapshot(net)
            star = '  <-- tốt nhất'
        print(f'[{tag}] epoch {ep:2d}/{epochs} | lr={lr_ep:.5f} | '
              f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
              f'val_loss={va_loss:.4f} val_acc={va_acc:.4f}{star}', flush=True)

    train_time = time.time() - t_start
    restore(net, best_snap)
    print(f'[{tag}] hoàn tất sau {train_time:.1f}s | epoch tốt nhất = {best_epoch} '
          f'| val_acc tốt nhất = {best_acc:.4f}')
    return hist, best_epoch, train_time


print('Đã định nghĩa hàm đánh giá và vòng lặp huấn luyện.')

### 9.1 Huấn luyện biến thể Baseline

Cấu hình: `padding = 0`, khởi tạo $\mathcal{N}(0,1) \times 0{,}01$, learning rate cố định
$10^{-3}$.

Điểm yếu đã biết trước của khởi tạo $\times 0{,}01$: với `fc1` có 1 152 đầu vào, độ lệch chuẩn của
tổng trọng số đầu ra chỉ khoảng $0{,}01\sqrt{1152} \approx 0{,}34$, nên tín hiệu bị co lại khi
truyền tới tầng cuối. Mạng vẫn học được nhờ Adam tự điều chỉnh bước theo độ lớn gradient, nhưng
những epoch đầu tiêu tốn phần lớn công sức chỉ để đưa biên độ trọng số về thang hợp lý.

In [ ]:
np.random.seed(RANDOM_SEED)
net_base = CNN(padding=0, init='scaled', seed=RANDOM_SEED, dtype=np.float32)
print(f'Baseline: {net_base.n_params():,} tham số | '
      f'sigma khởi tạo conv1 = {net_base.conv1.init_sigma}')
print(f'Huấn luyện trên tập con {len(X_tr):,} ảnh, kiểm định {len(X_va):,} ảnh, '
      f'{EPOCHS} epoch, batch {BATCH_SIZE}')
print('-' * 104)
hist_base, best_ep_base, time_base = train_model(
    net_base, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
    lr0=1e-3, lr_decay=None, tag='Baseline')

### 9.2 Huấn luyện biến thể Improved

Ba thay đổi so với Baseline, mỗi thay đổi có lý do riêng:

- **`padding = 1`.** Giữ nguyên kích thước không gian sau tích chập, nên thông tin ở viền ảnh
  không bị mất dần sau mỗi tầng. Với ảnh chỉ $32 \times 32$, việc mất hai điểm ảnh mỗi chiều mỗi
  tầng là đáng kể.
- **He Normal.** $\sigma = \sqrt{2/(C_{in}K^2)}$ giữ phương sai kích hoạt xấp xỉ không đổi khi
  truyền qua các tầng ReLU. Hệ số 2 bù cho việc ReLU triệt tiêu một nửa số kích hoạt.
- **Lịch giảm learning rate.** Bắt đầu ở $2 \times 10^{-3}$, nhân với $0{,}5$ sau mỗi 4 epoch.
  Bước lớn ở giai đoạn đầu để thoát nhanh khỏi vùng khởi tạo, bước nhỏ ở giai đoạn sau để tinh
  chỉnh quanh cực tiểu thay vì dao động quanh nó.

In [ ]:
np.random.seed(RANDOM_SEED)
net_impr = CNN(padding=1, init='he', seed=RANDOM_SEED, dtype=np.float32)
print(f'Improved: {net_impr.n_params():,} tham số')
print(f'  sigma He conv1 = {net_impr.conv1.init_sigma:.6f}  (sqrt(2/27)   = {np.sqrt(2/27):.6f})')
print(f'  sigma He conv2 = {net_impr.conv2.init_sigma:.6f}  (sqrt(2/144)  = {np.sqrt(2/144):.6f})')
print(f'  sigma He fc1   = {net_impr.fc1.init_sigma:.6f}  (sqrt(2/2048) = {np.sqrt(2/2048):.6f})')
print(f'Huấn luyện trên tập con {len(X_tr):,} ảnh, kiểm định {len(X_va):,} ảnh, '
      f'{EPOCHS} epoch, batch {BATCH_SIZE}')
print('-' * 104)
hist_impr, best_ep_impr, time_impr = train_model(
    net_impr, X_tr, y_tr, X_va, y_va, epochs=EPOCHS, batch=BATCH_SIZE,
    lr0=2e-3, lr_decay=0.5, decay_every=4, tag='Improved')

## 10. Đánh giá trên trọn vẹn 10 000 ảnh kiểm thử

Nhắc lại điều đã công bố ở đầu notebook: hai mô hình được huấn luyện trên tập con nhưng **đánh giá
trên toàn bộ tập kiểm thử gốc**. Nhờ vậy, mọi con số dưới đây so sánh trực tiếp được với kết quả
của PyTorch và Keras ở notebook 02, dù hai nhóm mô hình thấy lượng dữ liệu huấn luyện khác nhau.

In [ ]:
def full_metrics(net, X, y, name):
    t0 = time.time()
    loss, acc, pred, P = evaluate(net, X, y, batch=512, return_prob=True)
    prec, rec, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    cm = confusion_matrix(y, pred, labels=list(range(10)))
    per_class = (cm.diagonal() / cm.sum(axis=1)).astype(float)
    conf = P.max(axis=1)
    wrong = np.where(pred != y)[0]
    order = wrong[np.argsort(-conf[wrong])][:8]
    hce = [{'index': int(i), 'true': int(y[i]), 'pred': int(pred[i]),
            'confidence': float(conf[i])} for i in order]
    print(f'--- {name} trên {len(y):,} ảnh kiểm thử (suy luận {time.time()-t0:.1f}s) ---')
    print(f'  loss            = {loss:.6f}')
    print(f'  accuracy        = {acc:.6f}  ({acc*100:.2f}%)')
    print(f'  macro precision = {prec:.6f}')
    print(f'  macro recall    = {rec:.6f}')
    print(f'  macro F1        = {f1:.6f}')
    print(f'  số ảnh sai      = {len(wrong):,}')
    return dict(loss=float(loss), accuracy=float(acc), macro_precision=float(prec),
                macro_recall=float(rec), macro_f1=float(f1),
                confusion_matrix=cm.tolist(), per_class_accuracy=per_class.tolist(),
                high_conf_errors=hce, pred=pred)


res_base = full_metrics(net_base, X_te, y_te, 'Baseline (NumPy)')
print()
res_impr = full_metrics(net_impr, X_te, y_te, 'Improved (NumPy)')

gain = (res_impr['accuracy'] - res_base['accuracy']) * 100
gain_f1 = (res_impr['macro_f1'] - res_base['macro_f1']) * 100
err_drop = (1 - res_impr['accuracy']) / max(1e-12, (1 - res_base['accuracy']))
print()
print('=' * 78)
print(f'Cải thiện tuyệt đối về accuracy : {gain:+.2f} điểm phần trăm '
      f'({res_base["accuracy"]*100:.2f}% -> {res_impr["accuracy"]*100:.2f}%)')
print(f'Cải thiện tuyệt đối về macro F1 : {gain_f1:+.2f} điểm phần trăm')
print(f'Tỉ lệ lỗi còn lại so với Baseline: {err_drop:.3f} '
      f'(giảm {100*(1-err_drop):.1f}% số lỗi)')
print(f'Mốc đoán ngẫu nhiên              : 10,00%  '
      f'-> Baseline vượt mốc {res_base["accuracy"]*100-10:.2f} đpt, '
      f'Improved vượt mốc {res_impr["accuracy"]*100-10:.2f} đpt')
print(f'Số tham số                       : Baseline {net_base.n_params():,} | '
      f'Improved {net_impr.n_params():,} '
      f'(gấp {net_impr.n_params()/net_base.n_params():.2f} lần)')
print(f'Thời gian huấn luyện             : Baseline {time_base:.1f}s | '
      f'Improved {time_impr:.1f}s')
print('=' * 78)

In [ ]:
print('Báo cáo phân loại chi tiết của biến thể Improved trên 10 000 ảnh kiểm thử:')
print(classification_report(y_te, res_impr['pred'], digits=4, zero_division=0,
                            target_names=[f'{k}. {CLASS_VI[k]}' for k in range(10)]))

**Diễn giải bảng phân loại.** Các con số cụ thể được in ở ô trên. Điều cần đọc ra từ bảng không
chỉ là giá trị trung bình mà là **độ chênh lệch giữa các lớp**: những lớp có hình dáng đặc trưng
rõ ràng và nền tương đối đồng nhất (ô tô, tàu thủy, máy bay — thường có nền đường, nền biển, nền
trời) đạt điểm cao hơn hẳn nhóm động vật bốn chân (mèo, chó, hươu, ngựa), vốn giống nhau về cả
hình dáng tổng thể lẫn kết cấu bề mặt ở độ phân giải $32 \times 32$.

Đây chính là kiểm chứng định lượng cho dự đoán đã nêu ở notebook 00: mức độ khó không phân bố đều
giữa mười lớp, và mô hình nông này chưa đủ khả năng biểu diễn để tách các lớp gần nhau về mặt thị
giác.

## 11. Hình vẽ và nhận xét

In [ ]:
ep_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_axis, hist_base['train_loss'], 'o-', color='#2E86C1', label='Baseline train')
axes[0].plot(ep_axis, hist_base['val_loss'],  'o--', color='#2E86C1', alpha=0.55, label='Baseline val')
axes[0].plot(ep_axis, hist_impr['train_loss'], 's-', color='#C0392B', label='Improved train')
axes[0].plot(ep_axis, hist_impr['val_loss'],  's--', color='#C0392B', alpha=0.55, label='Improved val')
axes[0].set_title('Mất mát entropy chéo theo epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mất mát')
axes[0].legend(fontsize=9); axes[0].set_xticks(ep_axis)

axes[1].plot(ep_axis, np.array(hist_base['train_acc'])*100, 'o-', color='#2E86C1', label='Baseline train')
axes[1].plot(ep_axis, np.array(hist_base['val_acc'])*100,  'o--', color='#2E86C1', alpha=0.55, label='Baseline val')
axes[1].plot(ep_axis, np.array(hist_impr['train_acc'])*100, 's-', color='#C0392B', label='Improved train')
axes[1].plot(ep_axis, np.array(hist_impr['val_acc'])*100,  's--', color='#C0392B', alpha=0.55, label='Improved val')
axes[1].axvline(best_ep_base, color='#2E86C1', ls=':', lw=1.2)
axes[1].axvline(best_ep_impr, color='#C0392B', ls=':', lw=1.2)
axes[1].axhline(10.0, color='gray', ls='-.', lw=1.2, label='Mốc đoán ngẫu nhiên 10%')
axes[1].set_title('Độ chính xác theo epoch (đường chấm dọc = epoch tốt nhất)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Độ chính xác (%)')
axes[1].legend(fontsize=9, loc='lower right'); axes[1].set_xticks(ep_axis)

fig.suptitle('CNN NumPy thuần trên CIFAR-10: Baseline so với Improved\n'
             f'(huấn luyện trên tập con phân tầng {len(X_tr):,} ảnh, kiểm định {len(X_va):,} ảnh)',
             fontsize=14, y=1.04)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_scratch_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_scratch_curves.png')
print()
print(f'Mất mát validation cuối cùng    : Baseline {hist_base["val_loss"][-1]:.4f} | '
      f'Improved {hist_impr["val_loss"][-1]:.4f}')
print(f'Độ chính xác validation tốt nhất: Baseline {max(hist_base["val_acc"])*100:.2f}% '
      f'(epoch {best_ep_base}) | Improved {max(hist_impr["val_acc"])*100:.2f}% '
      f'(epoch {best_ep_impr})')
print(f'Khoảng cách train - val ở epoch cuối: '
      f'Baseline {(hist_base["train_acc"][-1]-hist_base["val_acc"][-1])*100:+.2f} đpt | '
      f'Improved {(hist_impr["train_acc"][-1]-hist_impr["val_acc"][-1])*100:+.2f} đpt')
print(f'Lịch learning rate của Improved : {sorted(set(hist_impr["lr"]), reverse=True)}')

**Diễn giải hình `fig_cifar10_scratch_curves.png`.** Ba điều đáng đọc từ hai bảng đồ thị:

1. **Đường Improved nằm dưới đường Baseline ở bảng mất mát và nằm trên ở bảng độ chính xác ngay
   từ những epoch đầu.** Khởi tạo He đưa mạng vào vùng tham số hợp lý ngay lập tức, trong khi
   Baseline phải tiêu vài epoch đầu chỉ để đưa biên độ trọng số lên thang đúng.
2. **Khoảng cách giữa đường train và đường validation.** Con số cụ thể được in ở ô trên. Trên
   CIFAR-10 với chỉ 10 000 ảnh huấn luyện và không có bất kỳ kỹ thuật chính quy hóa nào (không
   dropout, không tăng cường dữ liệu, không batch normalization), khoảng cách này mở rộng theo
   thời gian — dấu hiệu quá khớp điển hình. Đây chính là động cơ cho kiến trúc có dropout và batch
   normalization ở notebook 02.
3. **Hiệu ứng của lịch learning rate.** Ở các epoch ngay sau mỗi lần learning rate giảm một nửa,
   đường mất mát của Improved có bước hạ rõ rệt rồi phẳng dần: bước nhỏ hơn cho phép mạng tinh
   chỉnh quanh cực tiểu thay vì dao động quanh nó.

Cả hai đường đều nằm rất xa mốc 10% của phép đoán ngẫu nhiên, xác nhận mạng thực sự học được cấu
trúc thị giác chứ không chỉ khai thác thống kê nhãn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
tick_lbl = [f'{k}\n{CLASS_VI[k]}' for k in range(10)]
for ax, res, name in [(axes[0], res_base, 'Baseline'), (axes[1], res_impr, 'Improved')]:
    cm = np.array(res['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                annot_kws={'size': 7.5}, linewidths=0.4, linecolor='#DDDDDD',
                xticklabels=tick_lbl, yticklabels=[f'{k}. {CLASS_VI[k]}' for k in range(10)])
    ax.set_title(f'{name}: accuracy = {res["accuracy"]*100:.2f}%, '
                 f'{int(cm.sum() - np.trace(cm)):,} ảnh sai', fontsize=12)
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
    ax.tick_params(axis='x', labelsize=8); ax.tick_params(axis='y', labelsize=8)
fig.suptitle('Ma trận nhầm lẫn trên 10 000 ảnh kiểm thử CIFAR-10, CNN NumPy thuần',
             fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_scratch_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_scratch_confusion.png')
print()

cm_i = np.array(res_impr['confusion_matrix'])
off = [(cm_i[a, b], a, b) for a in range(10) for b in range(10) if a != b and cm_i[a, b] > 0]
off.sort(reverse=True)
print('Tám cặp nhầm lẫn nặng nhất của Improved (nhãn thật -> nhãn dự đoán):')
for n, a, b in off[:8]:
    print(f'  {CLASS_VI[a]:>9s} -> {CLASS_VI[b]:<9s} : {n:>4} ảnh  '
          f'({100*n/cm_i[a].sum():5.2f}% số ảnh lớp {CLASS_VI[a]})')
print()
# Nhóm ngữ nghĩa: phương tiện (0,1,8,9) và động vật (2..7)
veh, ani = [0, 1, 8, 9], [2, 3, 4, 5, 6, 7]
tot = cm_i.sum()
in_veh = int(cm_i[np.ix_(veh, veh)].sum())
in_ani = int(cm_i[np.ix_(ani, ani)].sum())
cross = tot - in_veh - in_ani
err_in_veh = in_veh - sum(cm_i[k, k] for k in veh)
err_in_ani = in_ani - sum(cm_i[k, k] for k in ani)
n_err = int(tot - np.trace(cm_i))
print('Phân rã lỗi theo nhóm ngữ nghĩa (phương tiện = máy bay, ô tô, tàu thủy, xe tải;')
print('động vật = chim, mèo, hươu, chó, ếch, ngựa):')
print(f'  tổng số ảnh sai                          : {n_err:,}')
print(f'  sai trong nội bộ nhóm phương tiện        : {err_in_veh:,} '
      f'({100*err_in_veh/n_err:.1f}% số lỗi)')
print(f'  sai trong nội bộ nhóm động vật           : {err_in_ani:,} '
      f'({100*err_in_ani/n_err:.1f}% số lỗi)')
print(f'  sai chéo giữa hai nhóm                   : {cross:,} '
      f'({100*cross/n_err:.1f}% số lỗi)')

**Diễn giải hình `fig_cifar10_scratch_confusion.png`.** Hai ma trận cho thấy đường chéo của
Improved đậm hơn Baseline ở gần như mọi lớp. Quan trọng hơn là **cấu trúc của phần ngoài đường
chéo**: khối lượng lỗi không rải đều mà tụ lại ở những ô nối các lớp gần nhau về mặt thị giác.

Phép phân rã theo nhóm ngữ nghĩa ở ô trên lượng hóa điều đó. Nếu mô hình đoán bừa, tỉ lệ lỗi chéo
giữa hai nhóm sẽ tỉ lệ thuận với kích thước nhóm. Tỉ lệ thực tế đo được lệch khỏi mức đó theo
hướng **lỗi tập trung trong nội bộ nhóm**, nghĩa là mạng đã học được sự phân biệt thô giữa
"phương tiện" và "động vật" trước khi học được sự phân biệt tinh bên trong mỗi nhóm. Đây là hành
vi hợp lý và cũng là thứ mà phân tích PCA không gian ẩn ở notebook `mlp_vs_cnn` sẽ kiểm chứng lại
bằng một góc nhìn khác.

In [ ]:
labels = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
vals_b = [res_base['accuracy'], res_base['macro_precision'],
          res_base['macro_recall'], res_base['macro_f1']]
vals_i = [res_impr['accuracy'], res_impr['macro_precision'],
          res_impr['macro_recall'], res_impr['macro_f1']]

xpos = np.arange(len(labels)); w = 0.36
fig, ax = plt.subplots(figsize=(11.5, 6.2))
b1 = ax.bar(xpos - w/2, np.array(vals_b)*100, w,
            label=f'Baseline (p=0, randn*0.01, lr cố định) — {net_base.n_params():,} tham số',
            color='#2E86C1', edgecolor='black', linewidth=0.6)
b2 = ax.bar(xpos + w/2, np.array(vals_i)*100, w,
            label=f'Improved (p=1, He Normal, lr giảm dần) — {net_impr.n_params():,} tham số',
            color='#C0392B', edgecolor='black', linewidth=0.6)
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.2f}', ha='center', fontsize=9)
for k in range(len(labels)):
    d = (vals_i[k] - vals_b[k]) * 100
    ax.annotate(f'{d:+.2f} đpt', xy=(xpos[k], max(vals_b[k], vals_i[k])*100 + 3.2),
                ha='center', fontsize=10, color='#196F3D', fontweight='bold')
ax.axhline(10.0, color='gray', ls='-.', lw=1.2, label='Mốc đoán ngẫu nhiên 10%')
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel('Giá trị (%)')
ax.set_ylim(0, max(max(vals_b), max(vals_i))*100 + 9)
ax.set_title('CNN NumPy thuần trên CIFAR-10: Baseline so với Improved\n'
             f'(đánh giá trên 10 000 ảnh kiểm thử; huấn luyện trên tập con {len(X_tr):,} ảnh)',
             fontsize=13)
ax.legend(fontsize=9, loc='upper right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_scratch_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_scratch_comparison.png')
print()
print(f"{'Chỉ số':<18}{'Baseline':>12}{'Improved':>12}{'Chênh lệch (đpt)':>20}")
print('-' * 62)
for k, lb in enumerate(labels):
    print(f'{lb:<18}{vals_b[k]*100:>11.2f}%{vals_i[k]*100:>11.2f}%'
          f'{(vals_i[k]-vals_b[k])*100:>19.2f}')
print('-' * 62)
print(f'{"Tham số":<18}{net_base.n_params():>12,}{net_impr.n_params():>12,}'
      f'{net_impr.n_params()-net_base.n_params():>19,}')
print(f'{"Thời gian (s)":<18}{time_base:>12.1f}{time_impr:>12.1f}'
      f'{time_impr-time_base:>19.1f}')

**Diễn giải hình `fig_cifar10_scratch_comparison.png`.** Mức chênh lệch tuyệt đối tính bằng điểm
phần trăm được ghi ngay trên mỗi cặp cột, đúng yêu cầu của đề bài. Bốn chỉ số biến động gần như
đồng pha, và điều đó có nguyên nhân rõ ràng: vì CIFAR-10 cân bằng tuyệt đối, `macro precision`,
`macro recall` và `macro F1` đều xấp xỉ `accuracy`. Khi dữ liệu mất cân bằng — như miền
`diabetes` trong cùng assignment — bốn chỉ số này tách nhau rất xa.

Cần trung thực về nguồn gốc của mức cải thiện. Improved hơn Baseline ở **ba** yếu tố cùng lúc, và
một trong ba là số tham số: bảng số ở trên cho thấy Improved có nhiều tham số hơn do đệm biên giữ
kích thước không gian lớn hơn trước khi làm phẳng. Vì vậy con số chênh lệch nên được đọc là "hiệu
quả tổng hợp của gói cải tiến", không phải "hiệu quả riêng của He Normal". Một thí nghiệm tách
riêng từng yếu tố sẽ cần bốn lần huấn luyện nữa, vượt ngân sách tính toán của bài tập này; báo
cáo ghi nhận đây là giới hạn của thí nghiệm thay vì bỏ qua nó.

## 12. Lưu kết quả trung gian

In [ ]:
def pack(res, hist, best_ep, ttime, net, epochs, framework):
    return {
        'framework': framework,
        'params': int(net.n_params()),
        'train_time_s': float(ttime),
        'epochs': int(epochs),
        'best_epoch': int(best_ep),
        'accuracy': res['accuracy'],
        'macro_precision': res['macro_precision'],
        'macro_recall': res['macro_recall'],
        'macro_f1': res['macro_f1'],
        'loss': res['loss'],
        'history': {'train_loss': hist['train_loss'], 'val_loss': hist['val_loss'],
                    'train_acc': hist['train_acc'],  'val_acc': hist['val_acc']},
        'confusion_matrix': res['confusion_matrix'],
        'per_class_accuracy': res['per_class_accuracy'],
        'high_conf_errors': res['high_conf_errors'],
    }


partial = {
    'numpy_baseline': pack(res_base, hist_base, best_ep_base, time_base, net_base,
                           EPOCHS, 'NumPy From Scratch (Baseline)'),
    'numpy_improved': pack(res_impr, hist_impr, best_ep_impr, time_impr, net_impr,
                           EPOCHS, 'NumPy From Scratch (Improved)'),
    'subset': {'n_train_subset': int(len(X_tr)), 'n_val_subset': int(len(X_va)),
               'n_test': int(len(X_te)), 'batch_size': BATCH_SIZE, 'epochs': EPOCHS},
    'preprocess': {'mean_per_channel': [float(v) for v in MEAN_C],
                   'std_per_channel': [float(v) for v in STD_C]},
    'gradient_check': {
        'eps': 1e-5, 'dtype': 'float64', 'n_checks': len(rows),
        'n_excluded_kink': int(n_kink),
        'kink_aware': True,
        'max_rel_error': float(max(r[5] for r in rows)),
        'rows': [{'layer': r[0], 'param': r[1], 'index': list(r[2]),
                  'analytic': r[3], 'numeric': r[4], 'rel_error': r[5]} for r in rows],
    },
}
out_path = os.path.join(REP_DIR, 'metrics_cifar10_scratch_partial.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(partial, f, ensure_ascii=False, indent=2)
print('Đã ghi', out_path, f'({os.path.getsize(out_path)/1024:.1f} KB)')
print()
for f in ['fig_cifar10_scratch_curves.png', 'fig_cifar10_scratch_confusion.png',
          'fig_cifar10_scratch_comparison.png']:
    p = os.path.join(FIG_DIR, f)
    print(f'  {f:42s} {os.path.getsize(p)/1024:7.1f} KB  tồn tại={os.path.exists(p)}')

## 13. Kết luận của notebook 01

Báo cáo đã cài đặt trọn vẹn một mạng tích chập hai chiều cho ảnh màu bằng NumPy thuần và kiểm
chứng tính đúng đắn của nó ở ba cấp độ độc lập nhau:

1. **Cấp phép toán** — đối chiếu đường `im2col` + GEMM với một cài đặt tham chiếu viết bằng vòng
   lặp tường minh trên dữ liệu ba kênh, và với một ví dụ tính tay.
2. **Cấp định tuyến gradient** — kiểm tra max pooling hai kênh cho đúng giá trị và đúng vị trí
   nhận gradient.
3. **Cấp toàn mạng** — kiểm chứng gradient bằng sai phân hữu hạn trung tâm trên cả bốn tensor
   tham số, ở độ chính xác `float64`, với dữ liệu CIFAR-10 thật.

Về kết quả thực nghiệm, gói cải tiến (đệm biên same, He Normal, lịch giảm learning rate) nâng độ
chính xác thêm một lượng được ghi rõ bằng điểm phần trăm ở mục 11. Cả hai biến thể đều vượt rất
xa mốc 10% của phép đoán ngẫu nhiên, nhưng còn cách xa mức trên 98% mà cùng kiến trúc này đạt được
trên MNIST. Khoảng cách đó chính là cái giá của nền cảnh tự nhiên, tư thế đa dạng và độ phân giải
thấp mà notebook 00 đã lượng hóa.

Notebook tiếp theo dựng hai mạng sâu hơn bằng PyTorch và Keras — có batch normalization và dropout
mà cài đặt NumPy này không có — rồi đối chiếu ba cách cài đặt trên cùng một tập kiểm thử.